# SuperMockLoad — many patches & SEDs

Combine patches (eventually the full sky) and work with SEDs.  One simulation
today; the API is unchanged as more patches / the full sky come online.

In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
from supermockload import SuperMock, available_patches, plots

# If the synthetic patches live elsewhere, set the data root once:
# import os; os.environ['SUPERMOCK_DATA'] = '/path/to/Mocks_v3_data'
# or pass root=... to SuperMock(...).

### Combine whatever patches are on disk

In [ ]:
patches = available_patches()
print('patches available:', patches)
sm = SuperMock(patches, downsample=200_000)   # per-patch downsample; concatenated
print('total galaxies:', f'{sm.n:,}', '| footprint:', round(sm.area_deg2, 1), 'deg^2')
print('per-patch counts:', {p: int((sm.patch_id == p).sum()) for p in patches})

`patch_id` tags every row, so you can split combined statistics back out by patch.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
plots.gsmf(sm, ax=ax[0])
plots.luminosity_function(sm, ax=ax[1])
fig.suptitle(f'combined patches {sm.patches}')
fig.tight_layout()

### SEDs

SEDs are ~1.6 TB/patch, so they are never loaded wholesale — pull them for
explicit rows.  They are `f_nu` [Jy] on the **rest** grid; observed wavelength
is `wave_rest * (1 + z)`.

In [ ]:
sm = SuperMock(3, downsample=100_000, seds=True)
rows = sm.sample(500, mask=(sm.redshift < 0.3) & (sm.logM_zobs > 10.5))
wave, sed = sm.seds(rows=rows)
print('SED block:', sed.shape, '| rest grid:', wave.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for k in range(6):
    wobs = wave * (1 + sm.redshift[rows[k]]) / 1e4      # micron
    good = sed[k] > 0
    ax.loglog(wobs[good], sed[k][good], lw=1)
ax.set(xlabel=r'$\lambda_{obs}\ [\mu m]$', ylabel=r'$f_\nu$ [Jy]',
       title='example painted SEDs')

### Snapshot a working subset (with SEDs) for instant reuse

In [ ]:
sm.save('multi_100k.snapshot.h5', surveys=('LSST', 'WISE'), seds_rows=rows)
snap = SuperMock.from_file('multi_100k.snapshot.h5')
w, s = snap.seds(rows=rows[:10])       # SED subset travels with the snapshot
print('reloaded snapshot:', snap, '| SED subset:', s.shape)